# Notebook 11 — Distillation recovery sweep + generalization tests

Notebook 10 showed offline-teacher distillation recovers 63% of the CoNLL F1 gap at Taylor 20% pruning. This notebook tests how the recipe scales and generalizes:

**Sparsity sweep** — run the same recipe at Taylor 20% (re-run as control) and **Taylor 50%** (much heavier damage, well past the knee). Compare recovery rates.

**Generalization tests** for each post-distill model:
- **CoNLL dev** (200 samples, disjoint from training) — same distribution, different sentences. Sanity check / direct recovery measure.
- **CoNLL test split** (200 samples) — different sentences, same domain, never seen during training or teacher caching. Tests within-task generalization.
- **MMLU** (full 2028 questions across 10 subjects) — completely different task. Tests cross-task transfer. The hypothesis is the LoRA recovers *next-token-prediction capability*, not specific facts, so MMLU should improve only modestly above pre-distill (or possibly not at all).

**Hyperparameters fixed across both sparsity levels** (matching notebook 10): r=16, alpha=32, full-precision AdamW, T=2, distillation α=0.5, 3 epochs × 500 cached samples = 1500 steps, MAX_TRAIN_LEN=320.

**Runtime budget**: Taylor calibration (~3 min, done once) + per-sparsity ~25 min (training + 3 evals) × 2 sparsities = ~55 min total.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, json, math, re, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

import config
from eval_mmlu import evaluate as mmlu_evaluate

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TILE_R, TILE_C = config.TILE_SIZES[0]
SPARSITY_LEVELS = [0.20, 0.50]

TEACHER_CACHE_PATH = "results/teacher_conll_logits.pt"
N_F1_SAMPLES = 200
MAX_NEW_TOKENS = 64

DISTILL_T = 2.0
DISTILL_ALPHA = 0.5
EPOCHS = 3
LR = 1e-4
MAX_TRAIN_LEN = 320

torch.set_float32_matmul_precision("high")
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} / {total/1e9:.2f} GB")

GPU free: 7.67 / 8.18 GB


## 1. Load model + cache MLP weights

In [2]:
print(f"loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=getattr(torch, config.TORCH_DTYPE),
    device_map=config.DEVICE,
)
model.eval()
model.config.use_cache = True

def is_mlp_name(name):
    return any(k in name for k in config.PRUNE_TARGETS_PATTERNS)

ORIGINAL_MLP_WEIGHTS = {}
for name, module in model.named_modules():
    if isinstance(module, nn.Linear) and is_mlp_name(name):
        ORIGINAL_MLP_WEIGHTS[name + ".weight"] = module.weight.data.detach().clone().cpu()
print(f"cached {len(ORIGINAL_MLP_WEIGHTS)} MLP matrices on CPU")

loading Qwen/Qwen2.5-3B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

cached 108 MLP matrices on CPU


## 2. CoNLL dev + test splits, prompt format, eval helpers

In [3]:
ds_dev  = load_dataset("eriktks/conll2003", split="validation", trust_remote_code=True)
ds_test = load_dataset("eriktks/conll2003", split="test",       trust_remote_code=True)
print(f"CoNLL dev: {len(ds_dev)}  test: {len(ds_test)}")

TAG_ID_TO_NAME = {0:"O",1:"B-PER",2:"I-PER",3:"B-ORG",4:"I-ORG",5:"B-LOC",6:"I-LOC",7:"B-MISC",8:"I-MISC"}

def entities_from_bio(tokens, tag_ids):
    entities = []; cur_words=None; cur_type=None
    for tok, tid in zip(tokens, tag_ids):
        tag = TAG_ID_TO_NAME[tid]
        if tag.startswith("B-"):
            if cur_words: entities.append((" ".join(cur_words), cur_type))
            cur_words=[tok]; cur_type=tag[2:]
        elif tag.startswith("I-") and cur_words and cur_type==tag[2:]:
            cur_words.append(tok)
        else:
            if cur_words:
                entities.append((" ".join(cur_words), cur_type)); cur_words=None; cur_type=None
    if cur_words: entities.append((" ".join(cur_words), cur_type))
    return entities

def format_entities(entities):
    return "none" if not entities else ", ".join(f"{txt} ({typ})" for txt, typ in entities)

FEW_SHOT = [
    ("Angela Merkel visited Berlin last Friday .", [("Angela Merkel","PER"),("Berlin","LOC")]),
    ("Apple Inc. acquired Beats Electronics in 2014 .", [("Apple Inc.","ORG"),("Beats Electronics","ORG")]),
    ("The Olympic Games will return to Paris in 2024 .", [("Olympic Games","MISC"),("Paris","LOC")]),
]
FEW_SHOT_PREFIX = "Extract named entities from each text. Answer with comma-separated 'entity (TYPE)' pairs, where TYPE is PER, ORG, LOC, or MISC. If there are no entities, write 'none'.\n\n"
for text, ents in FEW_SHOT:
    FEW_SHOT_PREFIX += f"Text: {text}\nEntities: {format_entities(ents)}\n\n"

def make_prompt(tokens):
    return FEW_SHOT_PREFIX + f"Text: {' '.join(tokens)}\nEntities:"
def make_full(tokens, entities):
    return make_prompt(tokens) + " " + format_entities(entities)

def parse_entities_from_completion(text):
    text = text.strip().split("\n")[0].strip()
    if not text or text.lower().startswith("none"):
        return []
    out = []
    for part in text.split(","):
        m = re.match(r"^(.+?)\s*\(([A-Z]+)\)\s*$", part.strip())
        if m:
            out.append((m.group(1).strip(), m.group(2).strip()))
    return out

def micro_f1(preds, refs):
    tp=fp=fn=0
    for p,r in zip(preds,refs):
        ps,rs = set(p), set(r)
        tp += len(ps&rs); fp += len(ps-rs); fn += len(rs-ps)
    P = tp/(tp+fp) if tp+fp else 0.0
    R = tp/(tp+fn) if tp+fn else 0.0
    Fs = 2*P*R/(P+R) if P+R else 0.0
    return P,R,Fs,tp,fp,fn

def eval_perplexity(model, examples, tag=""):
    total_nll, total_tokens = 0.0, 0
    for ex in tqdm(examples, desc=f"ppl[{tag}]"):
        toks = ex["tokens"]
        ents = entities_from_bio(toks, ex["ner_tags"])
        prompt = make_prompt(toks); full = make_full(toks, ents)
        prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"][0]
        full_ids = tokenizer(full, return_tensors="pt")["input_ids"][0]
        if len(full_ids) <= len(prompt_ids): continue
        with torch.no_grad():
            logits = model(full_ids.unsqueeze(0).to(config.DEVICE)).logits[0]
        ans_pred_start = len(prompt_ids) - 1
        ans_pred_end   = len(full_ids) - 1
        target_ids = full_ids[len(prompt_ids):].to(config.DEVICE)
        ans_logits = logits[ans_pred_start:ans_pred_end]
        log_probs = F.log_softmax(ans_logits.float(), dim=-1)
        nlls = -log_probs.gather(1, target_ids.unsqueeze(1)).squeeze(1)
        total_nll += nlls.sum().item(); total_tokens += nlls.numel()
    avg_nll = total_nll / max(total_tokens, 1)
    return {"perplexity": math.exp(avg_nll), "nll": avg_nll}

def eval_f1(model, examples, max_new=MAX_NEW_TOKENS, tag=""):
    preds, refs = [], []
    for ex in tqdm(examples, desc=f"F1[{tag}]"):
        toks = ex["tokens"]
        refs.append(entities_from_bio(toks, ex["ner_tags"]))
        prompt = make_prompt(toks)
        enc = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            out = model.generate(
                input_ids=enc["input_ids"].to(config.DEVICE),
                attention_mask=enc["attention_mask"].to(config.DEVICE),
                max_new_tokens=max_new, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        preds.append(parse_entities_from_completion(completion))
    P,R,Fs,tp,fp,fn = micro_f1(preds, refs)
    return {"precision":P,"recall":R,"f1":Fs,"tp":tp,"fp":fp,"fn":fn}

CoNLL dev: 3250  test: 3453


## 3. Subsets — disjoint train/dev-eval, separate test-eval

In [4]:
teacher_data = torch.load(TEACHER_CACHE_PATH, weights_only=False)
ppl_indices_used = teacher_data["ppl_indices"]
teacher_cache    = teacher_data["cache"]
DENSE_PPL = teacher_data["dense_ppl"]
DENSE_F1  = teacher_data["dense_f1"]
print(f"loaded teacher cache: {len(teacher_cache)} examples (top-K={teacher_data['k']})")
print(f"dense reference: PPL={DENSE_PPL:.3f}, F1={DENSE_F1:.3f}")

# Dev eval: disjoint from training
rng = np.random.RandomState(43)
all_dev = np.arange(len(ds_dev))
remaining_dev = np.setdiff1d(all_dev, np.array(ppl_indices_used))
f1_dev_indices = rng.choice(remaining_dev, size=N_F1_SAMPLES, replace=False)
f1_dev_examples = [ds_dev[int(i)] for i in f1_dev_indices]

# Test eval: from test split, never seen by teacher caching or training
rng_test = np.random.RandomState(44)
f1_test_indices = rng_test.choice(np.arange(len(ds_test)), size=N_F1_SAMPLES, replace=False)
f1_test_examples = [ds_test[int(i)] for i in f1_test_indices]

print(f"dev F1 eval: {len(f1_dev_examples)} sentences (disjoint from teacher cache)")
print(f"test F1 eval: {len(f1_test_examples)} sentences (from CoNLL test split)")

loaded teacher cache: 500 examples (top-K=64)
dense reference: PPL=1.725, F1=0.614
dev F1 eval: 200 sentences (disjoint from teacher cache)
test F1 eval: 200 sentences (from CoNLL test split)


## 4. C4 calibration + Taylor importance + masking utilities

In [5]:
N_CAL = 512
SEQ_LEN_CAL = 128

cal_ds = load_dataset(config.CALIBRATION_DATASET, config.CALIBRATION_SUBSET,
                     split="train", streaming=True, trust_remote_code=True)
cal_texts = []
for ex in cal_ds:
    t = ex.get("text", "")
    if len(t) > 50:
        cal_texts.append(t)
    if len(cal_texts) >= N_CAL: break
cal_encodings = [tokenizer(t, return_tensors="pt", max_length=SEQ_LEN_CAL, truncation=True)
                 for t in cal_texts]
print(f"calibration samples: {len(cal_encodings)}")

def get_component_type(name):
    for p in config.PRUNE_TARGETS_PATTERNS:
        if p in name: return p
    return "unknown"

def compute_taylor_importance(model, cal_encodings, tag=""):
    for p in model.parameters(): p.requires_grad_(False)
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2: p.requires_grad_(True)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    importance_gpu = {}
    def make_hook(pname):
        def hook(param):
            if param.grad is None: return
            taylor = (param.data * param.grad).abs().float()
            out_dim, in_dim = taylor.shape
            nr, nc = out_dim // TILE_R, in_dim // TILE_C
            tt = taylor[:nr*TILE_R, :nc*TILE_C].reshape(nr, TILE_R, nc, TILE_C).permute(0, 2, 1, 3)
            tile_scores = tt.reshape(nr, nc, -1).sum(dim=-1)
            if pname in importance_gpu: importance_gpu[pname] += tile_scores
            else: importance_gpu[pname] = tile_scores.clone()
            param.grad = None
        return hook
    handles = []
    for name, p in model.named_parameters():
        if is_mlp_name(name) and p.ndim == 2 and p.requires_grad:
            handles.append(p.register_post_accumulate_grad_hook(make_hook(name)))
    n = 0
    for enc in tqdm(cal_encodings, desc=f"taylor[{tag}]"):
        model.zero_grad(set_to_none=True)
        inputs = {k: v.to(config.DEVICE) for k, v in enc.items()}
        out = model(**inputs, labels=inputs["input_ids"])
        out.loss.backward(); n += 1
    for h in handles: h.remove()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    for p in model.parameters(): p.requires_grad_(False)
    importance = {name: (v / n).cpu().numpy() for name, v in importance_gpu.items()}
    del importance_gpu; gc.collect(); torch.cuda.empty_cache()
    return importance

def apply_taylor_masks_at(importance, sparsity, model_ref):
    comp_stats = {}
    for ct in config.PRUNE_TARGETS_PATTERNS:
        vals = np.concatenate([m.ravel() for n,m in importance.items() if ct in n])
        comp_stats[ct] = (vals.mean(), vals.std())
    norm = {}
    for name, m in importance.items():
        mu, sd = comp_stats[get_component_type(name)]
        norm[name] = (m - mu) / sd if sd > 1e-8 else np.zeros_like(m)
    masks = {}
    for name, m in norm.items():
        thr = float(np.percentile(m.ravel(), sparsity * 100))
        masks[name] = m < thr
    with torch.no_grad():
        for name, module in model_ref.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in masks:
                    mask = masks[pn]; w = module.weight.data
                    out_f, in_f = w.shape
                    nr, nc = out_f // TILE_R, in_f // TILE_C
                    for i in range(nr):
                        for j in range(nc):
                            if mask[i, j]:
                                w[i*TILE_R:(i+1)*TILE_R, j*TILE_C:(j+1)*TILE_C] = 0
    return masks

def restore_mlp_weights(model_ref):
    with torch.no_grad():
        for name, module in model_ref.named_modules():
            if isinstance(module, nn.Linear):
                pn = name + ".weight"
                if pn in ORIGINAL_MLP_WEIGHTS:
                    module.weight.data.copy_(ORIGINAL_MLP_WEIGHTS[pn].to(module.weight.device))

print("Taylor calibration on dense (one-time)...")
importance_taylor = compute_taylor_importance(model, cal_encodings, tag="dense")
print(f"  done, peak GPU {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

calibration samples: 512
Taylor calibration on dense (one-time)...


taylor[dense]: 100%|███████████████████████████████████████████████| 512/512 [03:24<00:00,  2.50it/s]


  done, peak GPU 7.01 GB


## 5. Distillation training utilities

In [6]:
from peft import LoraConfig, get_peft_model, TaskType

def make_lora_cfg():
    return LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.0, bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

def distill_step_loss(student_logits, teacher_topk_idx, teacher_topk_logits,
                      target_ids, ans_pred_start, ans_pred_end, T, alpha):
    student_ans = student_logits[ans_pred_start:ans_pred_end].float()
    log_p_full = F.log_softmax(student_ans, dim=-1)
    ce = -log_p_full.gather(1, target_ids.unsqueeze(1)).squeeze(1).mean()
    teacher_p     = F.softmax(teacher_topk_logits.float() / T, dim=-1)
    teacher_log_p = F.log_softmax(teacher_topk_logits.float() / T, dim=-1)
    student_log_p_T = F.log_softmax(student_ans / T, dim=-1)
    student_log_p_at_k = student_log_p_T.gather(1, teacher_topk_idx.long())
    kl = (teacher_p * (teacher_log_p - student_log_p_at_k)).sum(dim=-1).mean() * (T*T)
    return alpha * kl + (1.0 - alpha) * ce, ce.detach(), kl.detach()

def build_train_examples(teacher_cache, max_len=MAX_TRAIN_LEN):
    out = []
    for k, c in teacher_cache.items():
        if len(c["input_ids"]) > max_len: continue
        out.append({
            "input_ids":    c["input_ids"],
            "prompt_len":   c["prompt_len"],
            "topk_indices": c["topk_indices"].long(),
            "topk_logits":  c["topk_logits"].float(),
            "target_ids":   c["target_ids"],
        })
    return out

def train_distillation(model, train_examples, epochs=EPOCHS, lr=LR, T=DISTILL_T, alpha=DISTILL_ALPHA, tag=""):
    model.train()
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses_total, losses_ce, losses_kl = [], [], []
    t0 = time.time()
    for epoch in range(epochs):
        order = np.random.RandomState(42 + epoch).permutation(len(train_examples))
        pbar = tqdm(order, desc=f"distill[{tag}] e{epoch+1}/{epochs}")
        for idx in pbar:
            ex = train_examples[idx]
            input_ids = ex["input_ids"].unsqueeze(0).to(config.DEVICE)
            prompt_len = ex["prompt_len"]
            topk_idx = ex["topk_indices"].to(config.DEVICE)
            topk_log = ex["topk_logits"].to(config.DEVICE)
            target_ids = ex["target_ids"].to(config.DEVICE)
            out = model(input_ids=input_ids)
            ans_pred_start = prompt_len - 1
            ans_pred_end   = input_ids.shape[1] - 1
            n_ans = ans_pred_end - ans_pred_start
            if n_ans <= 0 or n_ans != topk_idx.shape[0]: continue
            loss, loss_ce, loss_kl = distill_step_loss(
                out.logits[0], topk_idx, topk_log, target_ids,
                ans_pred_start, ans_pred_end, T=T, alpha=alpha)
            loss.backward(); optim.step(); optim.zero_grad(set_to_none=True)
            losses_total.append(loss.item()); losses_ce.append(loss_ce.item()); losses_kl.append(loss_kl.item())
            if len(losses_total) % 50 == 0:
                pbar.set_postfix(L=f"{np.mean(losses_total[-50:]):.3f}",
                                  CE=f"{np.mean(losses_ce[-50:]):.3f}",
                                  KL=f"{np.mean(losses_kl[-50:]):.3f}")
    train_time = time.time() - t0
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.eval()
    return {
        "steps": len(losses_total), "time_s": train_time,
        "first50_total": float(np.mean(losses_total[:50])),
        "last50_total":  float(np.mean(losses_total[-50:])),
        "first50_ce":    float(np.mean(losses_ce[:50])),
        "last50_ce":     float(np.mean(losses_ce[-50:])),
        "first50_kl":    float(np.mean(losses_kl[:50])),
        "last50_kl":     float(np.mean(losses_kl[-50:])),
    }

## 6. Sweep loop — run for each sparsity, capture every metric

In [7]:
all_results = {}
train_examples = build_train_examples(teacher_cache)
print(f"distillation training set: {len(train_examples)} examples")

for sp in SPARSITY_LEVELS:
    print(f"\n{'='*72}")
    print(f"  Taylor {int(sp*100)}% — distillation recovery sweep")
    print(f"{'='*72}")

    # --- Reset state from any prior PEFT-wrapping, restore weights, apply mask
    # If a prior PEFT model exists, unload it back to the base. This depends on
    # the PeftModel.unload() method (returns underlying base) — see PEFT docs.
    if hasattr(model, "unload") and callable(getattr(model, "unload")):
        print("  unloading previous PEFT adapter...")
        model = model.unload()
        gc.collect(); torch.cuda.empty_cache()

    restore_mlp_weights(model)
    masks = apply_taylor_masks_at(importance_taylor, sp, model)
    actual = sum(m.sum() for m in masks.values()) / sum(m.size for m in masks.values())
    print(f"  applied Taylor {int(sp*100)}% mask (actual sparsity {actual*100:.1f}%)")

    # --- Pre-distillation eval on CoNLL dev only (cheap, gives baseline)
    model.eval(); model.config.use_cache = True
    pre_ppl = eval_perplexity(model, f1_dev_examples, tag=f"pre{int(sp*100)}")
    pre_f1  = eval_f1(model, f1_dev_examples, tag=f"pre{int(sp*100)}")
    print(f"  pre-distill: PPL={pre_ppl['perplexity']:.3f}  F1={pre_f1['f1']:.3f}")

    # --- Attach LoRA + train
    model = get_peft_model(model, make_lora_cfg())
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    train_stats = train_distillation(model, train_examples, tag=f"sp{int(sp*100)}")
    print(f"  trained {train_stats['steps']} steps in {train_stats['time_s']:.0f}s. "
          f"first50: total={train_stats['first50_total']:.3f}  last50: total={train_stats['last50_total']:.3f}")

    # --- Save adapter
    adapter_dir = f"results/lora_distill_taylor{int(sp*100)}"
    model.save_pretrained(adapter_dir)
    print(f"  saved adapter to {adapter_dir}")

    # --- Post-distill: CoNLL dev (disjoint), CoNLL test (different split), MMLU (different task)
    model.eval(); model.config.use_cache = True
    post_dev_ppl = eval_perplexity(model, f1_dev_examples,  tag=f"post-dev{int(sp*100)}")
    post_dev_f1  = eval_f1(model, f1_dev_examples,           tag=f"post-dev{int(sp*100)}")
    post_test_f1 = eval_f1(model, f1_test_examples,          tag=f"post-test{int(sp*100)}")
    print(f"  post-distill CoNLL dev:  PPL={post_dev_ppl['perplexity']:.3f}  F1={post_dev_f1['f1']:.3f}")
    print(f"  post-distill CoNLL test: F1={post_test_f1['f1']:.3f}  P={post_test_f1['precision']:.3f}  R={post_test_f1['recall']:.3f}")

    # MMLU eval — full 2028 questions across 10 subjects
    print("  evaluating MMLU...")
    mmlu_results = mmlu_evaluate(model, tokenizer, tag=f"distilled_taylor{int(sp*100)}")
    mmlu_acc = mmlu_results["overall"]["accuracy"]
    print(f"  post-distill MMLU: {mmlu_acc*100:.2f}% ({mmlu_results['overall']['correct']}/{mmlu_results['overall']['total']})")

    all_results[f"taylor_{int(sp*100)}"] = {
        "sparsity": sp,
        "pre_distill": {
            "conll_dev_ppl": pre_ppl["perplexity"], "conll_dev_nll": pre_ppl["nll"],
            "conll_dev_f1": pre_f1["f1"], "conll_dev_p": pre_f1["precision"], "conll_dev_r": pre_f1["recall"],
        },
        "training": train_stats,
        "post_distill": {
            "conll_dev_ppl": post_dev_ppl["perplexity"], "conll_dev_nll": post_dev_ppl["nll"],
            "conll_dev_f1": post_dev_f1["f1"], "conll_dev_p": post_dev_f1["precision"], "conll_dev_r": post_dev_f1["recall"],
            "conll_test_f1": post_test_f1["f1"], "conll_test_p": post_test_f1["precision"], "conll_test_r": post_test_f1["recall"],
            "mmlu_acc": mmlu_acc,
        },
    }

    # Free the optimizer / training state
    gc.collect(); torch.cuda.empty_cache()

distillation training set: 499 examples

  Taylor 20% — distillation recovery sweep
  applied Taylor 20% mask (actual sparsity 20.0%)


F1[pre20]: 100%|███████████████████████████████████████████████████| 200/200 [05:08<00:00,  1.54s/it]


  pre-distill: PPL=5.789  F1=0.097
trainable params: 22,560,768 || all params: 3,108,499,456 || trainable%: 0.7258


distill[sp20] e3/3: 100%|█████████████| 499/499 [01:35<00:00,  5.24it/s, CE=0.181, KL=0.571, L=0.376]


  trained 1497 steps in 286s. first50: total=1.907  last50: total=0.388
  saved adapter to results/lora_distill_taylor20


...

  trained 1497 steps in 281s. first50: total=7.893  last50: total=0.801
  saved adapter to results/lora_distill_taylor50


F1[post-test50]: 100%|█████████████████████████████████████████████| 200/200 [06:44<00:00,  2.02s/it]


  post-distill CoNLL dev:  PPL=1.714  F1=0.337
  post-distill CoNLL test: F1=0.324  P=0.484  R=0.243
  evaluating MMLU...


[distilled_taylor50] MMLU subjects: 100%|████████████████████████████| 10/10 [01:30<00:00,  9.01s/it]

  post-distill MMLU: 25.59% (519/2028)


## 7. Comparison summary

In [8]:
# MMLU dense baseline from notebook 4
MMLU_DENSE = 0.487

print("=" * 88)
print(f"  Distillation recovery sweep — Qwen 2.5 3B")
print("=" * 88)
print(f"  Dense reference:  CoNLL dev PPL={DENSE_PPL:.3f}  F1={DENSE_F1:.3f}   |  MMLU={MMLU_DENSE*100:.2f}%")
print()
header = f"  {'Variant':<20s}  {'CoNLL dev F1':>14s}  {'CoNLL test F1':>15s}  {'MMLU':>10s}  {'F1 recovery':>14s}"
print(header)
print("-" * 88)
for key in sorted(all_results.keys()):
    r = all_results[key]
    sp = r["sparsity"]
    pre_dev = r["pre_distill"]["conll_dev_f1"]
    post_dev = r["post_distill"]["conll_dev_f1"]
    post_test = r["post_distill"]["conll_test_f1"]
    mmlu = r["post_distill"]["mmlu_acc"]
    gap = DENSE_F1 - pre_dev
    rec = (post_dev - pre_dev) / gap * 100 if gap > 0 else 0
    print(f"  pre-distill T{int(sp*100)}%      {pre_dev:>13.3f}   {'-':>14s}     {'-':>9s}    {'-':>13s}")
    print(f"  post-distill T{int(sp*100)}%     {post_dev:>13.3f}   {post_test:>14.3f}     {mmlu*100:>8.2f}%   {rec:>11.1f}%")
    print()
print("=" * 88)

with open("results/distillation_sweep.json", "w") as f:
    json.dump({
        "model": MODEL_NAME,
        "hparams": {"r": 16, "alpha": 32, "T": DISTILL_T, "distill_alpha": DISTILL_ALPHA,
                    "epochs": EPOCHS, "lr": LR, "max_train_len": MAX_TRAIN_LEN},
        "dense_refs": {"conll_dev_ppl": DENSE_PPL, "conll_dev_f1": DENSE_F1, "mmlu_acc": MMLU_DENSE},
        "sweep": all_results,
    }, f, indent=2)
print("results/distillation_sweep.json saved")

  Distillation recovery sweep — Qwen 2.5 3B
  Dense reference:  CoNLL dev PPL=1.725  F1=0.614   |  MMLU=48.70%

  Variant                 CoNLL dev F1    CoNLL test F1        MMLU     F1 recovery
----------------------------------------------------------------------------------------
  pre-distill T20%              0.097                -             -                -
  post-distill T20%             0.471            0.316        26.68%          72.3%

  pre-distill T50%              0.000                -             -                -
  post-distill T50%             0.337            0.324        25.59%          54.8%

results/distillation_sweep.json saved
